In [42]:
# 2.1 Загрузка данных
import datetime
import os

import joblib
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import seaborn as sns
from autofeat import AutoFeatRegressor
from catboost import CatBoostRegressor
from dotenv import load_dotenv
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, RobustScaler, StandardScaler
from sqlalchemy import create_engine


SAVE_FIG_DIR = 'EDA_summary'

if not os.path.exists(SAVE_FIG_DIR):
    os.makedirs(SAVE_FIG_DIR)

pd.options.display.max_columns = 100
pd.options.display.max_rows = 64 

sns.set_style("white")
sns.set_theme(style="whitegrid") 

pd.set_option("display.float_format", "{:_.2f}".format)

train_dir_path = "models/logs/catboost_training"
os.makedirs(train_dir_path, exist_ok=True)

def create_connection():
    """ 
    Создание подключения к базе данных. 
    Возвращает:
    conn - объект подключения к базе данных
    """
    load_dotenv()
    host = os.environ.get('DB_DESTINATION_HOST')
    port = os.environ.get('DB_DESTINATION_PORT')
    db = os.environ.get('DB_DESTINATION_NAME')
    username = os.environ.get('DB_DESTINATION_USER')
    password = os.environ.get('DB_DESTINATION_PASSWORD')
    
    conn = create_engine(f'postgresql://{username}:{password}@{host}:{port}/{db}', connect_args={'sslmode':'require'})
    return conn

def get_data():
    """
    Загрузка данных из базы данных.
    Возвращает:
    data - DataFrame с данными
    """
    conn = create_connection()
    data = pd.read_sql('select * from clean_flat_target_price', conn, index_col='id')
    
    
    conn.dispose()

    return data
# загружаем данные в переменную data
data = get_data()
data.head()  # выводим первые 5 строк таблицы data

,floor,is_apartment,kitchen_area,living_area,rooms,studio,total_area,price,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator
id,,,,,,,,,,,,,,,,
38171,1,False,12.00,52.00,3,False,80.00,12900000,1999,4,55.83,37.35,2.70,301,14,True
38172,7,False,8.60,21.00,1,False,39.10,6650000,1977,4,55.78,37.83,2.64,381,14,True
38173,3,False,9.00,0.00,3,False,54.00,15500000,1963,1,55.68,37.59,2.48,80,5,False
38176,3,False,6.00,0.00,1,False,39.20,8700000,1990,4,55.68,37.72,2.64,127,9,True
38178,3,False,10.00,31.00,2,False,52.00,14000000,2002,4,55.65,37.40,2.74,119,15,True


In [48]:
print((data['studio']==False).value_counts())

studio
True    105965
Name: count, dtype: int64


In [44]:
data['rooms_mean_total_area']=data.groupby('rooms')['total_area'].transform('mean')
current_year = datetime.datetime.now().year
data['building_age'] = current_year - data['build_year']
data['total_rooms_per_floor']=data.groupby(['floor', 'rooms'])['rooms'].transform('count')
data['total_flats_per_floor'] = data.groupby(['floor'])['rooms'].transform('count')
kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
data["region_cluster"] = kmeans.fit_predict(data[["latitude", "longitude"]])

cat_features = [
    "rooms",
    "floor",
    "studio",
    "building_type_int",
    "has_elevator",
    "is_apartment",
    "region_cluster",
]
num_features = [
    "kitchen_area",
    "living_area",
    "total_area",
    "ceiling_height",
    "flats_count",
    "floors_total",
    "building_age",
    "rooms_mean_total_area",
    "total_rooms_per_floor",
    "total_flats_per_floor",
]
target = "price"
features = cat_features + num_features

preprocessor = ColumnTransformer(
    transformers=[
        #("num_one", RobustScaler(), num_features),
        ('num', StandardScaler(), num_features),
        ("cat", OrdinalEncoder(), cat_features),
    ],
    n_jobs=-1,
    remainder="drop",
)
X_transformed = preprocessor.fit_transform(data)




In [45]:
X_transformed_one=pd.DataFrame(X_transformed, columns=preprocessor.get_feature_names_out())
X_transformed_one.head()

,num__kitchen_area,num__living_area,num__total_area,num__ceiling_height,num__flats_count,num__floors_total,num__building_age,num__rooms_mean_total_area,num__total_rooms_per_floor,num__total_flats_per_floor,cat__rooms,cat__floor,cat__studio,cat__building_type_int,cat__has_elevator,cat__is_apartment,cat__region_cluster
0,0.99,1.62,1.62,-0.05,0.57,0.11,-0.69,1.32,-0.20,0.64,2.00,0.00,0.00,4.00,1.00,0.00,6.00
1,0.05,-0.46,-0.79,-0.37,1.17,0.11,0.38,-1.10,-0.15,-0.13,0.00,6.00,0.00,4.00,1.00,0.00,2.00
2,0.16,-1.87,0.09,-1.23,-1.06,-1.44,1.06,1.32,-0.02,0.96,2.00,2.00,0.00,1.00,0.00,0.00,5.00
3,-0.66,-1.87,-0.79,-0.37,-0.72,-0.75,-0.25,-1.10,0.72,0.96,0.00,2.00,0.00,4.00,1.00,0.00,7.00
4,0.44,0.21,-0.03,0.17,-0.77,0.28,-0.84,-0.04,1.55,0.96,1.00,2.00,0.00,4.00,1.00,0.00,9.00


In [46]:
X_transformed_two=pd.DataFrame(X_transformed, columns=features)
X_transformed_two.head()

,rooms,floor,studio,building_type_int,has_elevator,is_apartment,region_cluster,kitchen_area,living_area,total_area,ceiling_height,flats_count,floors_total,building_age,rooms_mean_total_area,total_rooms_per_floor,total_flats_per_floor
0,0.99,1.62,1.62,-0.05,0.57,0.11,-0.69,1.32,-0.20,0.64,2.00,0.00,0.00,4.00,1.00,0.00,6.00
1,0.05,-0.46,-0.79,-0.37,1.17,0.11,0.38,-1.10,-0.15,-0.13,0.00,6.00,0.00,4.00,1.00,0.00,2.00
2,0.16,-1.87,0.09,-1.23,-1.06,-1.44,1.06,1.32,-0.02,0.96,2.00,2.00,0.00,1.00,0.00,0.00,5.00
3,-0.66,-1.87,-0.79,-0.37,-0.72,-0.75,-0.25,-1.10,0.72,0.96,0.00,2.00,0.00,4.00,1.00,0.00,7.00
4,0.44,0.21,-0.03,0.17,-0.77,0.28,-0.84,-0.04,1.55,0.96,1.00,2.00,0.00,4.00,1.00,0.00,9.00


In [16]:
afc = AutoFeatRegressor(transformations=["abs"], feateng_steps=0, n_jobs=1, verbose=True)

In [12]:
import numpy as np
import pandas as pd

# Предполагая, что X_transformed - это pd.DataFrame
print("Количество NaN в X_transformed:", X_transformed.isna().sum().sum())
print("Количество бесконечностей в X_transformed:", np.isinf(X_transformed).sum().sum())

# Если X_transformed - numpy array, преобразуйте в DataFrame для удобства
if isinstance(X_transformed, np.ndarray):
    X_transformed_df = pd.DataFrame(X_transformed)
    print("Количество NaN:", X_transformed_df.isna().sum().sum())
    print("Количество бесконечностей:", np.isinf(X_transformed_df).sum().sum())

Количество NaN в X_transformed: 0
Количество бесконечностей в X_transformed: 0


In [35]:
print(X_transformed['floors_total'].info())
print(X_transformed['floors_total'].unique()[:50])
print(X_transformed['floors_total'].dtype)
print(X_transformed['floors_total'].min(), X_transformed['floors_total'].max())

<class 'pandas.core.series.Series'>
RangeIndex: 105965 entries, 0 to 105964
Series name: floors_total
Non-Null Count   Dtype  
--------------   -----  
105965 non-null  float64
dtypes: float64(1)
memory usage: 828.0 KB
None
[0.]
float64
0.0 0.0


In [29]:
y = data[target]
X = data.drop(columns=[target])
X_1=X_transformed[['kitchen_area','building_type_int','living_area','total_area','ceiling_height','flats_count','building_age','rooms_mean_total_area','total_rooms_per_floor','total_flats_per_floor']]
X_new = afc.fit_transform(X_1, y)

2025-08-30 13:27:47,965 INFO: [AutoFeat] The 0 step feature engineering process could generate up to 10 features.
2025-08-30 13:27:47,975 INFO: [AutoFeat] With 105965 data points this new feature matrix would use about 0.00 gb of space.
2025-08-30 13:27:47,996 WARNING: [feateng] no features generated for max_steps < 1.


2025-08-30 13:27:48,040 INFO: [featsel] Feature selection run 1/5


[featsel] Scaling data...done.


2025-08-30 13:27:50,543 INFO: [featsel] Feature selection run 2/5
2025-08-30 13:27:52,778 INFO: [featsel] Feature selection run 3/5
2025-08-30 13:27:55,072 INFO: [featsel] Feature selection run 4/5
2025-08-30 13:27:57,555 INFO: [featsel] Feature selection run 5/5
2025-08-30 13:27:59,905 INFO: [featsel] 10 features after 5 feature selection runs
c:\Users\Салтанат\Documents\Учеба в Яндексе\tests\.venv_mle\Lib\site-packages\autofeat\featsel.py:270: FutureWarning: Series.ravel is deprecated. The underlying array is already 1D, so ravel is not necessary.  Use `to_numpy()` for conversion to a numpy array instead.
  if np.max(np.abs(correlations[c].ravel()[:i])) < 0.9:
2025-08-30 13:27:59,972 INFO: [featsel] 8 features after correlation filtering
2025-08-30 13:28:00,667 INFO: [featsel] 8 features after noise filtering
2025-08-30 13:28:00,671 INFO: [AutoFeat] Final dataframe with 10 feature columns (0 new).
2025-08-30 13:28:00,671 INFO: [AutoFeat] Training final regression model.
2025-08-30 13